# RoSEHFL Kaggle Runner

## Required secrets:

- `GITHUB_TOKEN`: [Your GitHub Personal Access Token with repo permissions]
- `GDRIVE_CRED`: [Your Google Drive Credentials JSON content]
- `GDRIVE_FOLDER_ID`: [Your Google Drive Folder ID]

## Set the following values for customization

- `GITHUB_OWNER`: Your GitHub username or organization name
- `GITHUB_REPO`: The name of your GitHub repository
- `GITHUB_BRANCH`: The branch of your repository to clone
- `RUN_NAME`: A unique name for this run, used for saving results

In [ ]:
GITHUB_OWNER = 'SakiburRahman07'
GITHUB_REPO = 'RoSEHFL'
GITHUB_BRANCH = 'rose_q1s'

RUN_NAME = 'cmp_8_lenet5_fmnist_geant2010_n30_seed42'

In [ ]:
!apt-get update -y
!apt-get install -y rclone git

import textwrap
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'RoSEHFL'
REPO_DIR.mkdir(parents=True, exist_ok=True)
print('Workdir:', WORKDIR)

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
RCLONE_CONF_TEXT = secrets.get_secret("RCLONE_CONF")
GDRIVE_FOLDER_ID = secrets.get_secret('GDRIVE_FOLDER_ID')

print('Secrets loaded.')

In [ ]:
if REPO_DIR.exists():
    !rm -rf /kaggle/working/RoSEHFL

clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
!git clone --branch {GITHUB_BRANCH} --single-branch {clone_url} /kaggle/working/RoSEHFL

%cd /kaggle/working/RoSEHFL
!git rev-parse --abbrev-ref HEAD

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
cfg_dir = Path("/root/.config/rclone")
cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / "rclone.conf"
RCLONE_CONF_TEXT=RCLONE_CONF_TEXT.replace("\\n", "\n")
cfg_path.write_text(RCLONE_CONF_TEXT, encoding="utf-8")

!rclone config file
!rclone lsd gdrive:

In [ ]:
LOCAL_RUN_DIR = Path('/kaggle/working/RoSEHFL/results') / RUN_NAME
REMOTE_RUN_DIR = f'gdrive:RoSEHFL/kaggle_runs/{RUN_NAME}'

LOCAL_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)

print('Pulling previous state from Drive if present...')
!rclone copy {REMOTE_RUN_DIR} {LOCAL_RUN_DIR} --create-empty-src-dirs --transfers 8 --checkers 8 --ignore-errors || true

print('Local run dir:', LOCAL_RUN_DIR)
if LOCAL_RUN_DIR.exists():
    !find {LOCAL_RUN_DIR} -maxdepth 3 -type f | head -n 40

In [ ]:
!python -m scripts.run_comparison --resume --strategies fedavg fedprox cost_first data_first random share shapefl rose_q1s --model lenet5 --dataset fmnist --topology geant2010 --num-nodes 30 --kappa-e 1 --kappa-c 10 --kappa 50 --gamma-max 2800 --shapley-T 2 --shapley-K 3 --probe-size 256 --seed 42 --comparison-mode effective --output-dir {LOCAL_RUN_DIR}

In [ ]:
print('Syncing run directory to Drive...')
!rclone copy {LOCAL_RUN_DIR} {REMOTE_RUN_DIR} --create-empty-src-dirs --transfers 8 --checkers 8 --progress

print('Done.')
!rclone lsf {REMOTE_RUN_DIR} | head -n 100

In [ ]:
must_have = [
    LOCAL_RUN_DIR / 'run_status.json',
    LOCAL_RUN_DIR / 'comparison_results.json',
]

for p in must_have:
    print(p, 'OK' if p.exists() else 'MISSING')

strategy_dirs = LOCAL_RUN_DIR / 'strategies'
if strategy_dirs.exists():
    print('Strategies found:', [d.name for d in strategy_dirs.iterdir() if d.is_dir()])
    for sd in strategy_dirs.iterdir():
        if sd.is_dir():
            ckpt = sd / 'checkpoint.pkl'
            print(sd.name, 'checkpoint:', ckpt.exists())